# Develop acceptance loop

In [3]:
from sklearn.ensemble import IsolationForest
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataGenerator, CreditData, CreditDataSample
from berebasl.simulation.acceptance_loop import accept_based_on_top_percentent_of_arbitrary_var
from berebasl.estimation.basl import BASLPartialUnbiaser
from berebasl.estimation.bayesian_evaluation import BayesianMetric, batched_auroc
from berebasl.estimation.classifiers import TorchLogistic

In [ ]:
# Old sketch
if True:
    # Acceptance Loop

    stats: List[Dict[str, Union[float, int]]] = []
    models_state_dicts: List[Dict[str, Union[Dict[str, Any], str]]] = []

    for gen_nr in range(1, num_gens + 1):
        if gen_nr % 10 == 0:
            print("-- Iteration", f"{gen_nr}/{num_gens}:", credit_data.accepted_count, 
                "accepts and", credit_data.rejected_count, " rejects")
            
        ## Gather current statistics
        current_stats : dict = credit_data.data_stats()


        ## Get leakage-free data
        current_sample: CreditDataSample = credit_data.to_sample_dataset() # Ensure leakage-free data
        current_sample.manual_seed(initial_seed + gen_nr + 1) # internal rng handles seeds for splitting

        ## Accepts based scorecard
        ### Reset params to ensure no effect of last calculation
        classifier_accepts.reset_parameters_to_initial()
        classifier_accepts.fit(current_sample.features_labeled, current_sample.labels)

        classifier_accepts.eval()
        holdout_probs_bad_accepts_based = classifier_accepts.predict_proba(holdout_data.features)[..., 1]
        current_stats["auc_accepts"] = batched_auroc(holdout_probs_bad_accepts_based, holdout_data.default_flag).item()

        ## Oracle scorecard
        classifier_oracle.reset_parameters_to_initial()
        classifier_oracle.fit(credit_data.features, credit_data.default_flag) # Using explicitly all data

        classifier_oracle.eval()
        holdout_probs_bad_oracle = classifier_oracle.predict_proba(holdout_data.features)[..., 1]
        current_stats["auc_biased"] = batched_auroc(holdout_probs_bad_oracle, holdout_data.default_flag).item()

        ## Corrected scorecard
        augmented_sample: CreditDataSample = basl_unbiaser.basl_augment_sample(
            data=current_sample, 
            leave_orig_sample_untouched=True, 
            early_stop=True
        )
        basl_unbiaser.refit_model(which_one='strong', features=augmented_sample.features_labeled, labels=augmented_sample.labels)
        holdout_probs_basl = basl_unbiaser.predict_proba_model(which_one='strong', features=holdout_data.features)
        current_stats["auc_basl"] = batched_auroc(holdout_probs_basl, holdout_data.default_flag).item()

        stats.append(current_stats)

        if gen_nr == 1 or gen_nr % 10 == 0 or gen_nr == num_gens:
            models_state_dicts.append({
                "gen_round" : gen_nr,
                "accepts_model" : classifier_accepts.to_state_dict(),
                "oracle_model" : classifier_oracle.to_state_dict(),
                "basl_strong_model" : basl_unbiaser.strong_learner.to_state_dict()
            })

        if gen_nr < num_gens:
            ## Generate new data
            data_gen.manual_seed(initial_seed + gen_nr)
            # Let S:=sample_size
            feats_new_applicants, def_flag_new_applicants = data_gen.sample(sample_size, determinstic_mixture_weights) # [S, F], [S]
            with torch.no_grad():
                new_applicants_pred_def_probs = classifier_accepts.predict_proba(feats_new_applicants)[..., 1] # [S]
                new_applicants_accepted = new_applicants_pred_def_probs >= new_applicants_pred_def_probs.quantile(1-top_percent) # [S]

            max_accepts_allowed = round(sample_size*top_percent)
            currently_accepted = new_applicants_accepted.sum()
            if currently_accepted > max_accepts_allowed:
                count_to_flip = currently_accepted - max_accepts_allowed
                idx_accepted = torch.where(new_applicants_accepted)[0] # [currently_accepted,] 
                perm = torch.randperm(currently_accepted, generator=data_gen.rng, device=data_gen.device) 
                idx_to_flip = idx_accepted[perm[:count_to_flip]]
                new_applicants_accepted[idx_to_flip] = False
                
            credit_data.add_gen(feats_new_applicants, def_flag_new_applicants, new_applicants_accepted)
            

        break


## Checking reproducibility

Two simulations were generated with the same arguments on the same machine - only different amount of processors involved. They should therefore
contain pretty much the same data at the end.

In [33]:
sim_parent_dir = "../berebasl/data/simulations"
simulation_dirs = ['default_sim', 'replicate_default']

simulation_results = {}

for sim_dir in simulation_dirs:
    results_path = os.path.join(sim_parent_dir, sim_dir, "simulation_results.pt")
    simulation_results[sim_dir] = torch.load(results_path, map_location='cpu', weights_only=False)

In [34]:
sim1, sim2 = simulation_dirs
results_check = {}

def compare_credit_data(credit_data1: CreditData, credit_data2: CreditData) -> dict:
    if credit_data2.features.shape != credit_data1.features.shape:
        raise AssertionError("Features shape do not match, comparision not possible")
    results = {}

    results["features"] = {
        "all_equal" : torch.all(credit_data1.features==credit_data2.features).item(),
        "all_close" : torch.allclose(credit_data1.features, credit_data2.features)
    }

    results["gen_round_equal"] = torch.all(credit_data1.gen_round == credit_data2.gen_round).item()
    results["label_equal"] = torch.all(credit_data1.default_flag == credit_data2.default_flag).item()
    results["accepted_equal"] = torch.all(credit_data1.accepted == credit_data2.accepted).item()

    return results

compare_credit_data(simulation_results[sim1]["credit_data"], simulation_results[sim2]["credit_data"])

{'features': {'all_equal': True, 'all_close': True},
 'gen_round_equal': True,
 'label_equal': True,
 'accepted_equal': True}